# Python for Finance — Final practice session — instructor solutions

**Four exercises in examination conditions**

The statements and the data are identical to the student notebook. Each exercise is followed by one solution: the decisions that matter, the code, and the expected results for the given data.

*Candidate for instructor approval (patch P1).*

## Exercise 1 — Paying creditors in order of priority

**Context**

A small company has closed down. After selling everything it owned, the liquidator holds EUR 250 thousand in cash, which must now be distributed among the company's creditors.

For each creditor, we know:

- the creditor's name;
- the creditor's class: `"staff"`, `"public"`, `"bank"` or `"supplier"`;
- the amount the company owes to that creditor, in thousand EUR.

The information is stored in three lists. The first entry in each list belongs to the first creditor, the second entry belongs to the second creditor, and so on. Each creditor appears once. The list `class_order` gives the four classes in their order of priority.

The cash is distributed according to the following rules:

1. The classes are served one after the other, in the order of `class_order`.
2. If the cash still available is at least the total amount owed to a class, every creditor of that class is paid in full.
3. Otherwise, the cash still available is shared within that class in proportion to the amounts owed: every creditor of the class receives the same fraction of what it is owed, where

   ```text
   fraction = cash still available / total amount owed to the class
   ```

   The cash is then exhausted, and the creditors of the following classes receive nothing.

**Final objective**

By the end of the exercise, your program should produce:

- `paid`: a dictionary that maps every creditor to the amount it receives, in thousand EUR (`0.0` for a creditor who receives nothing);
- `recovery`: a dictionary that maps every class to the fraction of the total amount owed to that class that is actually paid (`1.0` for a class paid in full, `0.0` for a class that receives nothing).

In [ ]:
creditors = ["Bank North", "Rossi", "Tax office", "Paper Co", "Bank South",
             "Dubois", "Ink Ltd", "Social security"]
classes = ["bank", "staff", "public", "supplier", "bank",
           "staff", "supplier", "public"]
owed = [150.0, 20.0, 60.0, 90.0, 50.0, 30.0, 60.0, 40.0]

class_order = ["staff", "public", "bank", "supplier"]
cash = 250.0

**Solution.** The cash is distributed **class by class**, and what a creditor receives depends on its whole class: Bank North's payment depends on what Bank South is owed. So, for each class in `class_order`, we first add up what the class is owed, and only then compute **one fraction** for the whole class: `1.0` when the cash still available covers the class, `available / class_total` when it does not, and `0.0` for every later class, since `available` is then 0. Each creditor of the class receives `fraction * owed`, and the fraction itself is the `recovery` of the class.

In [ ]:
paid = {}
recovery = {}
available = cash

for class_name in class_order:
    class_total = 0.0
    for amount, creditor_class in zip(owed, classes):
        if creditor_class == class_name:
            class_total += amount

    if available >= class_total:
        fraction = 1.0
    else:
        fraction = available / class_total
    recovery[class_name] = fraction
    available = available - fraction * class_total      # 0.0 once the cash has run out

    for creditor, creditor_class, amount in zip(creditors, classes, owed):
        if creditor_class == class_name:
            paid[creditor] = fraction * amount

WATCH OUT for the class in which the cash runs out. Its fraction must be computed **once, before** anyone in the class is paid. 

**Expected results.**

| Class | Owed | Cash before | Fraction |
|---|---|---|---|
| staff | 50 | 250 | 1.0 |
| public | 100 | 200 | 1.0 |
| bank | 200 | 100 | 0.5 |
| supplier | 150 | 0 | 0.0 |

`paid` gives Rossi 20.0, Dubois 30.0, Tax office 60.0, Social security 40.0, Bank North 75.0, Bank South 25.0, Paper Co 0.0 and Ink Ltd 0.0. The whole 250.0 is distributed, and the order of priority, not the size of a claim, decides who is paid.

## Exercise 2 — Counting the votes of a shareholders' meeting

**Context**

The shareholders of a company vote on one resolution. Each shareholder has one vote per share, and the dictionary `register` gives the number of shares held by each shareholder.

Votes are sent electronically. The list `ballots` contains the ballots in the order in which they were received; each ballot is a tuple `(sender, choice)`.

The votes are counted according to the following rules:

1. A ballot is valid only if its sender appears in `register` and its choice is `"for"`, `"against"` or `"abstain"`. Invalid ballots are ignored.
2. A shareholder may send several ballots: only the shareholder's **last valid** ballot counts.
3. Quorum: the shareholders whose ballot counts, abstentions included, must together hold at least half of all the shares in `register`. Otherwise, the meeting cannot decide, and the outcome is `"no quorum"`.
4. If there is a quorum, the resolution is `"adopted"` if the shares voting `"for"` are more than half of the votes cast, and `"rejected"` otherwise. The votes cast are the shares voting `"for"` plus the shares voting `"against"`: abstentions are not votes cast.

**Final objective**

By the end of the exercise, your program should produce:

- `final_choice`: a dictionary that maps each shareholder whose ballot counts to the choice of that ballot;
- `shares_by_choice`: a dictionary that maps each of the three choices to the number of shares behind it (`0` if nobody made that choice);
- `outcome`: `"no quorum"`, `"adopted"` or `"rejected"`.

In [ ]:
register = {"Alba": 200, "Bruno": 75, "Chloe": 200,
            "Dev": 300, "Emma": 150, "Farid": 75}

ballots = [("Emma", "against"), ("Bruno", "for"), ("Gina", "for"),
           ("Chloe", "abstain"), ("Farid", "against"), ("Emma", "for"),
           ("Bruno", "yes")]

**Solution.** A dictionary keyed by shareholder applies rule 2 by itself: `final_choice[sender] = choice` adds a shareholder the first time and **replaces** the earlier choice when a later ballot arrives. The validity test must come **before** this assignment, so that an invalid ballot can never replace a valid one. The tally then goes through `final_choice`, not `ballots`, so that each shareholder counts once. The two tests use different totals: the quorum compares the shares **present**, abstentions included, with all the shares in `register` (1,000, including Alba and Dev, who did not vote); the majority compares the shares voting `"for"` with the **votes cast**, `"for"` plus `"against"` only. The two boundaries also differ: the quorum needs **at least** half, the majority **more than** half. Here the 500 shares present are exactly half of 1,000, so the quorum test must accept equality.

In [ ]:
valid_choices = ["for", "against", "abstain"]

final_choice = {}
for sender, choice in ballots:
    if sender in register and choice in valid_choices:
        final_choice[sender] = choice              # a later valid ballot replaces an earlier one

shares_by_choice = {"for": 0, "against": 0, "abstain": 0}
for shareholder, choice in final_choice.items():
    shares_by_choice[choice] += register[shareholder]

present = shares_by_choice["for"] + shares_by_choice["against"] + shares_by_choice["abstain"]
votes_cast = shares_by_choice["for"] + shares_by_choice["against"]

if present < sum(register.values()) / 2:
    outcome = "no quorum"
elif shares_by_choice["for"] > votes_cast / 2:
    outcome = "adopted"
else:
    outcome = "rejected"

WATCH OUT: each rule matters on these data. Keeping Emma's first ballot (`"against"`) gives `"rejected"`, and so does a guard such as `if sender not in final_choice:`, which keeps the first valid ballot instead of the last. Letting Bruno's invalid `"yes"` erase his valid ballot leaves 425 shares present, hence `"no quorum"`. Gina is not in `register`: without the membership test, `register["Gina"]` raises a `KeyError`.

**Expected results.** `final_choice` is `{"Emma": "for", "Bruno": "for", "Chloe": "abstain", "Farid": "against"}` and `shares_by_choice` is `{"for": 225, "against": 75, "abstain": 200}`. 500 of the 1,000 shares are present, exactly half, so the quorum is reached; 225 of the 300 votes cast are for, so `outcome` is `"adopted"`. Without Chloe's abstention, the meeting could not have decided.

## Exercise 3 — Protecting positions with a trailing stop

**Context**

A fund buys three stocks, A, B and C, at their closing prices on day 0, and protects each position with a **trailing stop**. A trailing stop follows the highest closing price that the stock has reached since the purchase: if the price falls far enough below that level, the position is sold.

The array `prices` contains the closing prices over seven days: one row per day (day 0 to day 6) and one column per stock (A, B, C). The array `shares` gives the number of shares bought of each stock, in the same order as the columns.

The rules are:

1. A stock is sold at the closing price of the **first** day on which that price is at most 75% of the highest closing price of the stock from day 0 up to that day (both included).
2. Once a stock is sold, its later prices no longer matter.
3. A stock that is never sold is valued at its closing price on day 6.

The **exit price** of a stock is its sale price, or its day-6 price if it is never sold. The profit or loss of a position is

```text
profit or loss = number of shares × (exit price − closing price on day 0)
```

**Final objective**

By the end of the exercise, your program should produce:

- `exit_day`: a list with, for each stock in column order, the day on which it is sold, or `None` if it is never sold;
- `exit_price`: a NumPy array of shape `(3,)` with the exit price of each stock;
- `pnl`: a NumPy array of shape `(3,)` with the profit or loss of each position;
- `total_pnl`: the profit or loss of the three positions together.

The input arrays `prices` and `shares` must remain unchanged.

In [ ]:
import numpy as np

# closing prices: one row per day (day 0 to day 6), one column per stock (A, B, C)
prices = np.array([[ 80, 40, 120],
                   [ 84, 36, 110],
                   [ 96, 31, 125],
                   [ 88, 29, 118],
                   [ 72, 33, 100],
                   [ 70, 35,  95],
                   [ 90, 38, 108]])
shares = np.array([50, 100, 20])      # shares bought of A, B, C

**Solution.** One column of `prices` is the history of one stock, so the stocks are treated one at a time, going down their column with `prices[day, j]`. Along a column we keep `highest`, updated **before** the test because the current day is part of the peak, and we stop with `break` at the first day that triggers the stop, so that a later day can never replace the first sale. The exit prices are written into a **separate** array, initialised with the day-6 prices, so that `prices` itself is never modified. The profit or loss needs no loop: arrays of shape `(3,)` are combined stock by stock.

In [ ]:
n_days, n_stocks = prices.shape
exit_day = []
exit_price = prices[-1].copy()            # day-6 prices: the exit price of a stock never sold

for j in range(n_stocks):
    highest = prices[0, j]
    sold_on = None
    for day in range(n_days):
        if prices[day, j] > highest:
            highest = prices[day, j]
        if prices[day, j] <= 0.75 * highest:
            sold_on = day
            exit_price[j] = prices[day, j]
            break
    exit_day.append(sold_on)

pnl = shares * (exit_price - prices[0])
total_pnl = pnl.sum()

WATCH OUT for the boundary and the reference level. A reaches 96 on day 2, and 75% of 96 is exactly 72, its price on day 4. With `<` instead of `<=`, A would be sold on day 5 at 70; with a stop measured from the purchase price (60), never. `highest` must also start again for each stock. Writing the exit prices into `prices[-1]` itself, without `.copy()`, gives the same four results but changes the day-6 row of `prices`, which the statement forbids; a list of exit prices converted with `np.array(...)` is equally correct.

**Expected results.** `exit_day` is `[4, 3, None]` and `exit_price` is `[72, 29, 108]`: B is sold on day 3 at 29, below 75% of 40 = 30, and C never falls to 93.75, 75% of its peak of 125. `pnl` is `[-400, -1100, -240]` and `total_pnl` is −1,740: the stops limit the losses, they do not prevent them. `prices` and `shares` are unchanged.

## Exercise 4 — Reconciling positions with the custodian

**Context**

Every evening, the back office of an investment fund checks that the number of shares the fund believes it holds in each stock agrees with the custodian, the bank that keeps the shares on the fund's behalf.

The fund does not store its positions: they are rebuilt from the trade blotter `trades`, which has one row per trade, with:

- the trade ID;
- the ticker of the stock;
- the side, `"buy"` or `"sell"`;
- the number of shares traded;
- the status of the trade, `"settled"` or `"cancelled"`.

The custodian keeps the fund's shares in two accounts. Its report, `custodian_report`, has one row per account and stock, with the number of shares held.

The reconciliation follows these rules:

1. A cancelled trade never took place: it is ignored.
2. The fund position in a stock is the number of shares bought minus the number of shares sold in that stock.
3. The custodian position in a stock is the number of shares held in all of the fund's accounts together.
4. A stock that is missing on one side has a position of 0 on that side: a stock with no trade that took place has a fund position of 0, and a stock that does not appear in `custodian_report` has a custodian position of 0.
5. A stock is a **break** when its custodian position differs from its fund position.

**Final objective**

By the end of the exercise, your program should produce:

- `fund_position`: a Series, indexed by ticker, with the fund position in every stock that appears in at least one trade that took place;
- `difference`: a Series, indexed by ticker, with the custodian position minus the fund position, for every stock that appears in `fund_position` or in `custodian_report`;
- `breaks`: the part of `difference` that concerns the stocks that are breaks.

In [ ]:
import numpy as np
import pandas as pd

trades = pd.DataFrame({
    "trade_id": ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9"],
    "ticker": ["AAA", "BBB", "AAA", "CCC", "DDD", "BBB", "DDD", "CCC", "AAA"],
    "side": ["buy", "buy", "sell", "buy", "buy", "sell", "sell", "buy", "buy"],
    "quantity": [100, 50, 30, 80, 40, 20, 40, 60, 10],
    "status": ["settled", "settled", "settled", "cancelled", "settled",
               "settled", "settled", "settled", "cancelled"],
})

custodian_report = pd.DataFrame({
    "account": ["A-01", "A-01", "A-01", "A-02", "A-02"],
    "ticker": ["AAA", "BBB", "CCC", "AAA", "EEE"],
    "quantity": [50, 50, 60, 20, 25],
})

**Solution.** The two sides describe the same quantity, the shares held in each stock, but at different levels of detail: one row per **trade** for the fund, one row per **account and stock** for the custodian. Each side is therefore first reduced to one number per ticker, the fund side after removing the cancelled trades and giving sells a negative sign. The comparison then goes through **every** ticker that appears on either side, takes 0 for a side on which the ticker is missing (rule 4), and builds `difference` as a Series from the resulting values.

In [ ]:
settled = trades.loc[trades["status"] != "cancelled"].copy()
settled["signed_quantity"] = np.where(settled["side"] == "buy", settled["quantity"], -settled["quantity"])
fund_position = settled.groupby("ticker")["signed_quantity"].sum()
custodian_position = custodian_report.groupby("ticker")["quantity"].sum()

tickers = []                                   # every ticker that appears on either side
for position in [fund_position, custodian_position]:
    for ticker in position.index:
        if ticker not in tickers:
            tickers.append(ticker)

gaps = []
for ticker in tickers:                         # rule 4: a missing side counts as 0
    fund = fund_position.loc[ticker] if ticker in fund_position.index else 0
    held = custodian_position.loc[ticker] if ticker in custodian_position.index else 0
    gaps.append(held - fund)

difference = pd.Series(gaps, index=tickers)
breaks = difference.loc[difference != 0]

WATCH OUT for the stocks present on one side only. Subtracting the two Series directly, `custodian_position - fund_position`, matches them by ticker but gives `NaN` for DDD (no custodian row) and EEE (no trade). Since `NaN != 0` is `True`, DDD would be reported as a break and EEE's break would have no size. Rule 4 must be applied explicitly, either as above or with `custodian_position.sub(fund_position, fill_value=0)` (Session 2, "Index alignment"): both are correct.

**Expected results.**

| Ticker | Fund position | Custodian position | `difference` |
|---|---|---|---|
| AAA | 70 | 70 | 0 |
| BBB | 30 | 50 | 20 |
| CCC | 60 | 60 | 0 |
| DDD | 0 | 0 | 0 |
| EEE | 0 | 25 | 25 |

`breaks` contains BBB (20) and EEE (25). BBB looks as if the sale of 20 shares in T6 had not reached the custodian; the 25 EEE shares appear in no trade of the fund.